# Faz 4 — Görüntüleme Modülü

**Girdi:** bir IFC dosyası &nbsp;→&nbsp; **Çıktı:** 3D model + graph + IFC metni, **senkron seçim**.

Bir görünümde bir öğeye tıklayınca diğer ikisinde de ilgili öğe renk değiştirir; "Seçimi kaldır" sıfırlar. Graph node'ları sürüklenebilir.

> Bu notebook adım adım ilerler: (1) IFC yükle, (2) ortak modele parse et, (3) statik önizleme, (4) IFC satır vurgusu, (5) tam interaktif arayüz.

## 0) Kurulum — `src/` yolunu ekle

In [ ]:
import sys, glob
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = REPO / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('repo :', REPO)
print('src  :', SRC)

## 1) Girdi IFC
Varsa `data/baseline_ifc/` içindeki en yeni baseline'ı kullan; yoksa Faz 1.1 motoruyla üret.

In [ ]:
from ifc_gen.baseline.rule_based import build_baseline
from ifc_gen import paths

found = sorted(glob.glob(str(paths.baseline_ifc_dir() / 'baseline_rule_*.ifc')))
if found:
    ifc_path = found[-1]
    print('Mevcut baseline kullanılıyor:', ifc_path)
else:
    _, ifc_path, meta = build_baseline()
    print('Yeni baseline üretildi:', ifc_path)

## 2) Ortak modele parse et
`ViewerModel`: her eleman için GlobalId (ortak anahtar), 3D mesh, IFC satırları ve graph kenarlarını birlikte tutar. Senkron seçimin temeli budur.

In [ ]:
from viewer.model import load_viewer_model
vm = load_viewer_model(ifc_path)
print(f'Şema: {vm.schema} | eleman: {len(vm.elements)} | kenar: {len(vm.edges)}')
print()
for e in vm.elements.values():
    nv = 0 if e.verts is None else len(e.verts)
    print(f'  {e.ifc_type:10s} {e.name:12s} #{e.ifc_id:<4d} verts={nv:<4d} satır={len(e.line_ids)}')

In [ ]:
# Graph kenarları (ilişki tipleri)
nm = {e.ekey: e.name for e in vm.elements.values()}
for s, d, r in vm.edges[:15]:
    print(f'  {nm[s]:12s} --{r}--> {nm[d]}')
print(f'  ... toplam {len(vm.edges)} kenar')

## 3) Statik önizleme (her ortamda görünür)
Bir elemanı seçili kabul edip 3D + graph'ta nasıl senkron renklendiğini gösterir.

In [ ]:
import matplotlib.pyplot as plt
from viewer import static

secili = next(e.ekey for e in vm.elements.values() if e.ifc_type == 'IfcWall')
fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(121, projection='3d'); static.plot_3d(vm, secili, ax=ax1)
ax2 = fig.add_subplot(122); static.plot_graph(vm, secili, ax=ax2)
plt.tight_layout(); plt.show()

## 4) IFC satır vurgusu
Seçili elemanın STEP dosyasındaki ilgili satırları (kendisi + yerleşim/temsil).

In [ ]:
idxs = vm.highlight_line_indices(secili)
print(f'{vm.elements[secili].name} -> vurgulanan satırlar: {[i+1 for i in idxs]}')
for i in idxs:
    print(f'  {i+1:>5}  {vm.ifc_lines[i]}')

## 5) Tam interaktif arayüz (canlı Jupyter)
Aşağıdaki hücre üç görünümü birlikte açar:

- **Graph** node'una tıkla → o node, 3D'deki eleman ve IFC satırları renk değiştirir.
- **3D** modelde bir mesh'e tıkla → graph ve IFC senkron güncellenir.
- **IFC öğe** açılır listesinden seç → diğer ikisi güncellenir.
- **Seçimi kaldır** → tüm renkler sıfırlanır.
- Graph node'ları fareyle **sürüklenebilir**.

> Not: İnteraktif görünüm için Jupyter (Lab/Notebook) çekirdeği ve `ipywidgets` etkin olmalı. `labels` parametresiyle ileride ihlal/decoy/compliant renkleri eklenecek.

In [ ]:
from viewer.render import InteractiveViewer
viewer = InteractiveViewer(vm)   # labels={} -> baseline'da ihlal yok
viewer.show()

In [ ]:
# Programatik seçim de mümkün (test/otomasyon):
viewer.select(secili)
print('Seçili:', vm.elements[viewer.selected].name)
# viewer.select(None)  # seçimi kaldır